In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/dataset_egrul.csv", encoding='cp1251', sep=";")
df.info()
df.head(100)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   ID          500000 non-null  int64 
 1   TYPE        500000 non-null  object
 2   BEGIN_DATE  500000 non-null  object
 3   END_DATE    321455 non-null  object
 4   SEGMENT     500000 non-null  object
dtypes: int64(1), object(4)
memory usage: 19.1+ MB


,ID,TYPE,BEGIN_DATE,END_DATE,SEGMENT
0,1,ИП,14.12.2022,NaN,ТОРГОВЛЯ
1,2,ИП,29.08.2017,NaN,НАУКА
2,3,ЮЛ,19.03.2008,NaN,ЗДРАВООХРАНЕНИЕ
3,4,ЮЛ,15.12.2009,NaN,ТОРГОВЛЯ
4,5,ЮЛ,13.06.2013,NaN,НАУКА
...,...,...,...,...,...
95,96,ИП,16.11.2023,NaN,СТРОИТЕЛЬСТВО
96,97,ИП,21.12.2004,NaN,НАУКА
97,98,ИП,10.05.2018,15.03.2022,ТОРГОВЛЯ
98,99,ИП,11.05.2004,08.02.2021,ТОРГОВЛЯ


In [ ]:
df = df.replace(
    ['', ' ', 'NULL', 'NaN'],
    pd.NA
)


In [ ]:
df.head(100)

,ID,TYPE,BEGIN_DATE,END_DATE,SEGMENT
0,1,ИП,14.12.2022,NaN,ТОРГОВЛЯ
1,2,ИП,29.08.2017,NaN,НАУКА
2,3,ЮЛ,19.03.2008,NaN,ЗДРАВООХРАНЕНИЕ
3,4,ЮЛ,15.12.2009,NaN,ТОРГОВЛЯ
4,5,ЮЛ,13.06.2013,NaN,НАУКА
...,...,...,...,...,...
95,96,ИП,16.11.2023,NaN,СТРОИТЕЛЬСТВО
96,97,ИП,21.12.2004,NaN,НАУКА
97,98,ИП,10.05.2018,15.03.2022,ТОРГОВЛЯ
98,99,ИП,11.05.2004,08.02.2021,ТОРГОВЛЯ


In [ ]:
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
)


In [ ]:
df['begin_date'] = pd.to_datetime(
    df['begin_date'],
    format='%d.%m.%Y',
    errors='coerce'
)

df['end_date'] = pd.to_datetime(
    df['end_date'],
    format='%d.%m.%Y',
    errors='coerce'
)


In [ ]:
df[['begin_date', 'end_date']].isna().sum()


,0
begin_date,0
end_date,178545


In [ ]:
FUTURE_DATE = pd.Timestamp('2099-12-31')

df['end_date'] = df['end_date'].fillna(FUTURE_DATE)


In [ ]:
df['id'] = df['id'].astype(int)
df['type'] = df['type'].astype('category')
df['segment'] = df['segment'].astype('category')


In [ ]:
df.head(50)

,id,type,begin_date,end_date,segment,begin_year,end_year
0,1,ИП,2022-12-14,2099-12-31,ТОРГОВЛЯ,2022,2099
1,2,ИП,2017-08-29,2099-12-31,НАУКА,2017,2099
2,3,ЮЛ,2008-03-19,2099-12-31,ЗДРАВООХРАНЕНИЕ,2008,2099
3,4,ЮЛ,2009-12-15,2099-12-31,ТОРГОВЛЯ,2009,2099
4,5,ЮЛ,2013-06-13,2099-12-31,НАУКА,2013,2099
5,6,ИП,2020-11-10,2099-12-31,СТРОИТЕЛЬСТВО,2020,2099
6,7,ЮЛ,2008-09-25,2099-12-31,СЕЛЬСКОЕ ХОЗЯЙСТВО,2008,2099
7,8,ЮЛ,2015-09-17,2018-10-12,ТОРГОВЛЯ,2015,2018
8,9,ИП,2013-04-25,2013-08-16,ТРАНСПОРТ,2013,2013
9,10,ЮЛ,2008-04-18,2022-06-02,ТОРГОВЛЯ,2008,2022


In [ ]:
df.to_csv(
    "tochka.csv",
    index=False,
    encoding="utf-8"
)


In [ ]:
df.dtypes


,0
id,int64
type,category
begin_date,datetime64[ns]
end_date,datetime64[ns]
segment,category
begin_year,int32
end_year,int32


In [ ]:
liquidations_count = (df['end_date'] != '2099-12-31').sum()
print(liquidations_count)

321455


In [ ]:
closed_df = df[(df['end_date'] != '2099-12-31')].copy()
print(f"Всего записей: {len(df)}")
print(f"Ликвидировано: {len(closed_df)}")


Всего записей: 500000
Ликвидировано: 321455


In [ ]:
closed_df['lifetime_days'] = (
    closed_df['end_date'] - closed_df['begin_date']
).dt.days


In [ ]:
mean_lifetime_all = closed_df['lifetime_days'].mean()

print(f"Среднее время жизни бизнеса (дни): {mean_lifetime_all:.1f}")
print(f"Среднее время жизни бизнеса (годы): {mean_lifetime_all / 365:.2f}")


Среднее время жизни бизнеса (дни): 1842.4
Среднее время жизни бизнеса (годы): 5.05


In [ ]:
import numpy as np
df['DATE_CHECK'] = np.where(df['end_date'] >= df['begin_date'], 1, 0)
print(df[['begin_date', 'end_date', 'DATE_CHECK']].head(50))

   begin_date   end_date  DATE_CHECK
0  2022-12-14 2099-12-31           1
1  2017-08-29 2099-12-31           1
2  2008-03-19 2099-12-31           1
3  2009-12-15 2099-12-31           1
4  2013-06-13 2099-12-31           1
5  2020-11-10 2099-12-31           1
6  2008-09-25 2099-12-31           1
7  2015-09-17 2018-10-12           1
8  2013-04-25 2013-08-16           1
9  2008-04-18 2022-06-02           1
10 2014-03-07 2021-09-02           1
11 2011-08-05 2013-10-18           1
12 2014-04-11 2019-06-06           1
13 2018-05-29 2099-12-31           1
14 2014-02-05 2099-12-31           1
15 2013-09-17 2020-05-22           1
16 2019-02-19 2099-12-31           1
17 2019-09-12 2020-12-25           1
18 2021-04-26 2022-06-07           1
19 2019-03-20 2020-06-15           1
20 2004-02-18 2004-02-28           1
21 2013-11-21 2020-08-21           1
22 2016-11-02 2020-02-06           1
23 2015-10-13 2016-11-21           1
24 2010-05-11 2011-12-22           1
25 2018-11-26 2021-01-25           1
2

In [ ]:
count_fake = (df['DATE_CHECK'] == 0).sum()
print(count_fake)

22


In [ ]:
df['DATE_ERROR'] = df['end_date'] < df['begin_date']
df['lo'] = df['end_date'] - df['begin_date']

# Находим все некорректные записи
incorrect_records = df[df['DATE_ERROR'] == True]

In [ ]:
print(incorrect_records[['id']].to_string(index=False)) #получили 22 id, где дата ликвидации бизнеса раньше, чем дата регистрации

    id
 41017
 43805
 56703
 76265
 89860
 94591
109008
149569
153861
161475
177721
193972
225489
239291
254613
303379
319948
373281
423654
441914
451286
452510
